In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("Ecommerce Case Study") \
    .getOrCreate()

print("Spark Version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 06:02:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.6


In [2]:
import os

for root, dirs, files in os.walk("data"):
    print(root)
    for f in files:
        print("   ", f)

In [3]:
customers_df = spark.read.csv("customers/*.csv", header=True, inferSchema=True)

products_df = spark.read.csv("products/*.csv", header=True, inferSchema=True)

orders_df = spark.read.csv("orders/*.csv", header=True, inferSchema=True)

order_items_df = spark.read.csv("order_items/*.csv", header=True, inferSchema=True)

returns_df = spark.read.csv("returns/*.csv", header=True, inferSchema=True)

customers_df.show(5)

products_df.show(5)

orders_df.show(5)

print("Customers:", customers_df.count())
print("Products :", products_df.count())
print("Orders   :", orders_df.count())
print("Returns  :", returns_df.count())

26/06/16 06:03:10 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electroni

In [4]:
sales_by_category = (
    order_items_df
    .join(products_df, "product_id")
    .withColumn(
        "sales_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy("category")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy(desc("total_sales"))
)

sales_by_category.show(truncate=False)

[Stage 26:===================================================>    (11 + 1) / 12]

+--------------+--------------+
|category      |total_sales   |
+--------------+--------------+
|Beauty        |7.626693059E8 |
|Home & Kitchen|7.5813887328E8|
|Books         |7.4649077835E8|
|Toys          |7.446190723E8 |
|Electronics   |7.4426650411E8|
|Sports        |7.4333886813E8|
|Clothing      |7.4192279457E8|
+--------------+--------------+



In [5]:
top_customers = (
    orders_df
    .join(order_items_df, "order_id")
    .join(customers_df, "customer_id")
    .withColumn(
        "purchase_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        round(sum("purchase_amount"), 2).alias("total_purchase")
    )
    .orderBy(desc("total_purchase"))
    .limit(10)
)

top_customers.show(truncate=False)

[Stage 34:===================================================>    (12 + 1) / 13]

+-----------+--------------+--------------+
|customer_id|customer_name |total_purchase|
+-----------+--------------+--------------+
|93094      |Customer_93094|181569.68     |
|64560      |Customer_64560|169060.4      |
|23289      |Customer_23289|161573.8      |
|52275      |Customer_52275|153364.79     |
|61218      |Customer_61218|153067.55     |
|52034      |Customer_52034|152680.05     |
|40442      |Customer_40442|151037.32     |
|60528      |Customer_60528|148691.95     |
|84830      |Customer_84830|148363.84     |
|82593      |Customer_82593|148281.04     |
+-----------+--------------+--------------+



In [6]:
latest_year = (
    orders_df
    .select(max(year("order_date")).alias("max_year"))
    .collect()[0]["max_year"]
)

monthly_sales = (
    orders_df
    .join(order_items_df, "order_id")
    .filter(year("order_date") == latest_year)
    .withColumn(
        "sales_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(month("order_date").alias("month"))
    .agg(
        round(sum("sales_amount"), 2).alias("monthly_sales")
    )
    .orderBy("month")
)

monthly_sales.show()

[Stage 46:===================================================>    (12 + 1) / 13]

+-----+--------------+
|month| monthly_sales|
+-----+--------------+
|    1|4.4457777576E8|
|    2| 4.153661442E8|
|    3|4.4362824541E8|
|    4|4.2782097434E8|
|    5|4.4481061895E8|
|    6|4.3170515406E8|
|    7|4.4367051912E8|
|    8|4.4109517702E8|
|    9|4.3107152608E8|
|   10|4.4136378931E8|
|   11|4.3362336404E8|
|   12|4.4271290835E8|
+-----+--------------+



In [7]:
total_orders_category = (
    order_items_df
    .join(products_df, "product_id")
    .groupBy("category")
    .agg(count("*").alias("total_orders"))
)

returned_orders_category = (
    returns_df
    .join(order_items_df, "order_id")
    .join(products_df, "product_id")
    .groupBy("category")
    .agg(count("*").alias("returned_orders"))
)

return_percentage = (
    total_orders_category
    .join(
        returned_orders_category,
        "category",
        "left"
    )
    .fillna(0)
    .withColumn(
        "return_percentage",
        round(
            (col("returned_orders") /
             col("total_orders")) * 100,
            2
        )
    )
)

return_percentage.show(truncate=False)

[Stage 55:==============>                                          (3 + 9) / 12]

+--------------+------------+---------------+-----------------+
|category      |total_orders|returned_orders|return_percentage|
+--------------+------------+---------------+-----------------+
|Home & Kitchen|434034      |43418          |10.0             |
|Sports        |424412      |42530          |10.02            |
|Electronics   |425896      |42601          |10.0             |
|Clothing      |427607      |42660          |9.98             |
|Books         |427086      |42809          |10.02            |
|Beauty        |430547      |43194          |10.03            |
|Toys          |430418      |43382          |10.08            |
+--------------+------------+---------------+-----------------+



In [8]:
from pyspark.sql.functions import count, max as _max

df = (
    customers_df
    .join(orders_df, "customer_id")
)

payment_counts = (
    df.groupBy("state", "payment_mode")
      .agg(count("*").alias("order_count"))
)

payment_counts.show()

max_counts = (
    payment_counts
    .groupBy("state")
    .agg(_max("order_count").alias("max_order_count"))
)

max_counts.show()

result = (
    payment_counts
    .join(max_counts, "state")
    .filter(
        payment_counts.order_count ==
        max_counts.max_order_count
    )
    .select(
        "state",
        "payment_mode",
        "order_count"
    )
)

result.show()

+-----+----------------+-----------+
|state|    payment_mode|order_count|
+-----+----------------+-----------+
|   FL|     Net Banking|      19629|
|   NC|     Net Banking|      19596|
|   GA|      Debit Card|      19733|
|   OH|             UPI|      19930|
|   MI|Cash on Delivery|      20205|
|   IL|Cash on Delivery|      20498|
|   OH|     Net Banking|      20351|
|   TX|     Credit Card|      19874|
|   IL|             UPI|      20359|
|   NY|             UPI|      20108|
|   NY|     Net Banking|      20229|
|   TX|             UPI|      20065|
|   GA|     Credit Card|      19722|
|   TX|      Debit Card|      19988|
|   MI|     Net Banking|      20116|
|   NY|Cash on Delivery|      20208|
|   CA|      Debit Card|      20024|
|   IL|     Net Banking|      20404|
|   CA|     Credit Card|      19979|
|   OH|     Credit Card|      19794|
+-----+----------------+-----------+
only showing top 20 rows



+-----+---------------+
|state|max_order_count|
+-----+---------------+
|   MI|          20416|
|   CA|          20246|
|   NC|          19596|
|   IL|          20498|
|   WA|          20244|
|   OH|          20351|
|   NY|          20369|
|   TX|          20065|
|   GA|          20041|
|   FL|          20010|
+-----+---------------+



[Stage 72:===============>                                         (3 + 8) / 11]

+-----+----------------+-----------+
|state|    payment_mode|order_count|
+-----+----------------+-----------+
|   NC|     Net Banking|      19596|
|   IL|Cash on Delivery|      20498|
|   OH|     Net Banking|      20351|
|   TX|             UPI|      20065|
|   GA|     Net Banking|      20041|
|   WA|             UPI|      20244|
|   CA|             UPI|      20246|
|   FL|      Debit Card|      20010|
|   NY|      Debit Card|      20369|
|   MI|      Debit Card|      20416|
+-----+----------------+-----------+



In [12]:
customer_analysis = (
    orders_df
    .join(order_items_df, "order_id")
    .join(products_df, "product_id")
    .join(customers_df, "customer_id")
    .withColumn(
        "amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        countDistinct("category").alias("categories_bought"),
        round(sum("amount"), 2).alias("total_spent")
    )
    .filter(
        (col("categories_bought") >= 5) &
        (col("total_spent") > 100000)
    )
)

customer_analysis.show()

26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:08:45 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------+--------------+-----------------+-----------+
|customer_id| customer_name|categories_bought|total_spent|
+-----------+--------------+-----------------+-----------+
|      52297|Customer_52297|                7|  107812.68|
|      26241|Customer_26241|                7|   121047.8|
|      41157|Customer_41157|                7|   105187.1|
|      18149|Customer_18149|                7|   101780.7|
|      46060|Customer_46060|                7|  115805.04|
|      90203|Customer_90203|                7|  109729.63|
|      97920|Customer_97920|                7|  117591.17|
|      28078|Customer_28078|                7|  115138.03|
|      67631|Customer_67631|                7|  106734.26|
|      68399|Customer_68399|                7|   119426.7|
|      60620|Customer_60620|                7|  110953.51|
|      76094|Customer_76094|                7|   119256.5|
|      17867|Customer_17867|                7|  105736.33|
|      27963|Customer_27963|                7|  115995.7

In [14]:
# Q8 find the top 3 products by revenue within each category

product_revenue = (
    order_items_df
    .join(products_df,"product_id")
    .groupBy("category","product_id","product_name")
    .agg(
        sum(
            col("quantity")*col("selling_price")
        ).alias("revenue")
    )
)
product_revenue.show()

windows_spec = (
    Window
    .partitionBy("category")
    .orderBy(desc("revenue"))
)


top_products = (
    product_revenue
    .withColumn(
        "rank",dense_rank().over(windows_spec)
    )
    .filter(col("rank")<=3)
    .orderBy("category","rank")
)

top_products.show()

+-----------+----------+-------------+------------------+
|   category|product_id| product_name|           revenue|
+-----------+----------+-------------+------------------+
|   Clothing|      1280| Product_1280|         185289.69|
|Electronics|     24507|Product_24507| 90113.40000000001|
|Electronics|     37498|Product_37498|          53721.08|
|     Beauty|     18640|Product_18640|         107917.53|
|     Sports|     45205|Product_45205|         114789.16|
|     Beauty|      1893| Product_1893|          18459.02|
|Electronics|     17156|Product_17156|         158318.22|
|       Toys|     44426|Product_44426|101340.68999999999|
|Electronics|     21030|Product_21030|105387.98999999999|
|Electronics|     49242|Product_49242|          72795.13|
|     Sports|     11448|Product_11448|          74467.66|
|     Sports|      1300| Product_1300|164779.80999999997|
|       Toys|     37099|Product_37099|          83755.54|
|   Clothing|     15493|Product_15493|101753.73000000001|
|Electronics| 

[Stage 125:====>                                                  (1 + 11) / 12]

+--------------+----------+-------------+------------------+----+
|      category|product_id| product_name|           revenue|rank|
+--------------+----------+-------------+------------------+----+
|        Beauty|     44016|Product_44016|         277567.99|   1|
|        Beauty|     14849|Product_14849|274894.20000000007|   2|
|        Beauty|       786|  Product_786|272174.69999999995|   3|
|         Books|     35314|Product_35314| 296468.7799999999|   1|
|         Books|     28311|Product_28311|286757.72000000003|   2|
|         Books|     37479|Product_37479|         276736.71|   3|
|      Clothing|      7025| Product_7025|         293821.97|   1|
|      Clothing|      1560| Product_1560|         288474.09|   2|
|      Clothing|     31322|Product_31322|         282241.17|   3|
|   Electronics|      6719| Product_6719|         299113.87|   1|
|   Electronics|     23519|Product_23519|289561.72000000003|   2|
|   Electronics|     38170|Product_38170|288875.23000000004|   3|
|Home & Ki